# Spike V8: omurga eğitimi (kol S ve kol M)

**Ne yapar:** `configs/v2_pretrain_v8.yaml` ile bir hayvan için iki omurgayı eğitir (§12.8 Aşama 1b, §12.21):
- `multiday` (kol **M**): ÖN günleri + Day-0, hepsi Day-0 istatistikleriyle → Day-0 ince ayarı → W_0.
- `day0` (kol **S**): yalnız Day-0. İki kol aynı augmentation ve **aynı GPU türüyle** eğitilir.

`KIP = "ayar"`: seed 100–102 (ayar koşusunun girdisi). `KIP = "rapor"`: seed 0–4 (`--report-test`).
`FINETUNE = "readout"`: M kolunun omurga-donuk ince ayar varyantı (yalnız ayarda karşılaştırma için; S kolu
yeniden eğitilmez).

**Önce:** `v8_veri.ipynb` bitmiş olmalı (Drive'ım → `Spike_colab/cache/`). Çalışma zamanı **GPU (L4)**.
Her seed ve kol ayrı süreç (KR-043); bağlantı koparsa yeniden **Tümünü çalıştır** (biten adımlar atlanır,
yarım kalan `--resume` ile sürer).

## 1. Ayarlar

In [ ]:
HAYVAN = "indy"                    # indy | loco — her hayvan kendi defter sekmesinde (hayvan yalıtımı)
KIP = "ayar"                       # ayar (seed 100–102) | rapor (seed 0–4)
FINETUNE = "full"                  # full | readout (yalnız ayarda)
DRIVE_KLASOR = "/content/drive/MyDrive/Spike_colab"
DUMAN = False                      # True: 1 epoch'luk hızlı deneme (çıktı /content/duman, Drive'a yazılmaz)
SEEDLER = [100, 101, 102] if KIP == "ayar" else [0, 1, 2, 3, 4]
AD = "v2_pretrain_v8" + ("" if FINETUNE == "full" else "_ro")

## 2. GPU kontrolü

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU (L4)"

## 3. Drive, kod ve önbellek

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil, zipfile, pathlib, glob, subprocess, time
D = pathlib.Path(DRIVE_KLASOR)
assert (D / "spike_kod.zip").exists(), f"{D / 'spike_kod.zip'} yok: notebooks/README.md (V8) adım 1–3"
KOD = pathlib.Path("/content/Spike")
if KOD.exists():
    shutil.rmtree(KOD)
KOD.mkdir()
for f in ["spike_kod.zip", "spike_veri.zip"]:
    if (D / f).exists():
        with zipfile.ZipFile(D / f) as z:
            z.extractall(KOD)
os.chdir(KOD)
CACHE = KOD / "data/primate/cache"
CACHE.mkdir(parents=True, exist_ok=True)
DCACHE = D / "cache"                                   # v8_veri.ipynb'nin ürettiği önbellekler
DCACHE.mkdir(exist_ok=True)
for p in DCACHE.glob("*.pt"):
    if not (CACHE / p.name).exists():
        shutil.copy(p, CACHE / p.name)
surum = json.loads((KOD / "KOD_SURUMU.json").read_text(encoding="utf-8"))
print("kod:", surum["commit"], "(kirli)" if surum["dirty"] else "(temiz)", "| paket:", surum["paket_zamani"])
print("önbellek:", len(list(CACHE.glob("*.pt"))), "dosya")
ENV = {**os.environ, "PYTHONFAULTHANDLER": "1", "PYTHONIOENCODING": "utf-8"}

## 4. Bağımlılıklar ve hızlı test

In [ ]:
import torch as _t
_onceki = _t.__version__
!pip -q install pyyaml numpy pytest h5py neurobench==2.3.0
!python -c "import torch; from neurobench.metrics.workload import R2; print('neurobench tamam | torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
print('kurulumdan önceki torch:', _onceki, '(üstteki satırla aynı olmalı; farklıysa Çalışma zamanı → Oturumu yeniden başlat)')
!python -m pytest tests/v2/test_v8_data.py tests/v2/test_v8_train.py -q -x

## 5. Eğitim

In [ ]:
LOG_DIR = "/content/duman" if DUMAN else str(D / "runs_v8")
os.makedirs(LOG_DIR, exist_ok=True)
ISARET = pathlib.Path(LOG_DIR) / "_bitti"
ISARET.mkdir(exist_ok=True)
GRUPLAR = [("multiday", ["multiday"])] + ([("day0", ["day0"])] if FINETUNE == "full" else [])


def klasorler(seed):
    return sorted(glob.glob(f"{LOG_DIR}/*_{AD}_{HAYVAN}_seeds{seed}"), reverse=True)


def calistir(seed, etiket, asamalar):
    isaret = ISARET / f"{AD}_{HAYVAN}_s{seed}_{etiket}"
    if isaret.exists():
        print(f"[atla] {HAYVAN} seed {seed} {etiket} (bitmiş)")
        return
    for deneme in (1, 2, 3):
        havuz = klasorler(seed)
        cmd = ["python", "experiments/v2_pretrain.py", "--config", "configs/v2_pretrain_v8.yaml",
               "--primates", HAYVAN, "--seeds", str(seed), "--stages", *asamalar,
               "--set", f"experiment.log_dir={LOG_DIR}", f"experiment.name={AD}", f"multiday.finetune={FINETUNE}"]
        if KIP == "rapor":
            cmd.append("--report-test")
        if havuz:
            cmd += ["--resume", *havuz]
        if DUMAN:
            cmd += ["--epochs", "1"]
        log = pathlib.Path(LOG_DIR) / f"log_{AD}_{HAYVAN}_s{seed}_{etiket}_{deneme}.txt"
        t0 = time.time()
        with open(log, "w", encoding="utf-8") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=ENV).returncode
        yeni = klasorler(seed)
        if yeni:                                              # KV8: kod sürümü
            shutil.copy(KOD / "KOD_SURUMU.json", pathlib.Path(yeni[0]) / "KOD_SURUMU.json")
        print(f"{HAYVAN} seed {seed} {etiket} deneme {deneme}: çıkış {rc}, {(time.time() - t0) / 60:.1f} dk → "
              f"{yeni[0] if yeni else '-'}", flush=True)
        if rc == 0:
            isaret.touch()
            return
        print(log.read_text(encoding="utf-8")[-3000:])
    raise RuntimeError(f"{HAYVAN} seed {seed} {etiket}: 3 denemede de başarısız; log: {LOG_DIR}")


for s in SEEDLER:
    for etiket, asamalar in GRUPLAR:
        calistir(s, etiket, asamalar)
print("BİTTİ:", HAYVAN, KIP, FINETUNE, SEEDLER)

## 6. Özet

Bak: Day-0 val R² iki kolda yakın mı (Indy ≈ 0,77)? Donuk Day-k val R² (`multiday` ve `day0` satırları) M'de daha yüksek mi?

In [ ]:
for s in SEEDLER:
    for k in klasorler(s):
        k = pathlib.Path(k)
        print(k.name, sorted(p.name for p in k.glob("*.pt")))
        t = k / "table.md"
        if t.exists():
            print(t.read_text(encoding="utf-8"))